# Cuantización int8 de la red neuronal


Conversión de la red MLP de la versión v0.1 (`red.keras`, float32) a TensorFlow Lite con cuantización entera de 8 bits, para su ejecución en el ESP32-S3 con TensorFlow Lite Micro. Se evalúa el modelo cuantizado sobre las mismas 52 semanas de 2017 del notebook 02 y se exporta como arreglo de C++ para el firmware.


In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


import tempfile
import numpy as np
import pandas as pd
import tensorflow as tf
import keras

H = 7 # días de la semana a predcir(Horizonte de pronostico)
VENTANA = 28 # días recientes que la red recibe como entrada
ANIO = 364 # desfase de 52 semana

VERSION_ORIGEN = "v0.1"   # versión de la red float32
VERSION = "v0.2"          # versión del modelo cuantizado
MODELOS_DIR = f"../models/{VERSION}"
FIRMWARE_DIR = "../../firmware/include"
os.makedirs(MODELOS_DIR, exist_ok=True)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| motor:", keras.backend.backend())

TensorFlow 2.21.0 | Keras 3.15.1 | motor: tensorflow


## 1. Datos y variables de entrada

Se reconstruyen las variables de entrada con las mismas funciones del notebook 02, para que el modelo cuantizado se evalúe sobre exactamente las mismas filas.

In [2]:
raw = pd.read_csv("../data/train.csv", parse_dates=["date"])
raw["unique_id"] = "tienda" + raw.store.astype(str) + "_prod" + raw.item.astype(str)
tabla = raw.pivot(index="unique_id", columns="date", values="sales")

fechas = tabla.columns
Y = tabla.to_numpy(dtype="float32")
print(f"Y: {Y.shape[0]} series x {Y.shape[1]} días")

Y: 500 series x 1826 días


In [3]:
def idx(fecha):
    return fechas.get_loc(pd.Timestamp(fecha))

primer_origen = ANIO + VENTANA - 1
orig_train = np.arange(primer_origen, idx("2015-12-31") - H + 1)
orig_val = np.arange(idx("2015-12-31"), idx("2016-12-31") - H + 1)
orig_test = np.arange(idx("2017-01-01"), idx("2017-12-31") - H + 1, H)


def armar_pistas(Y, t):
    """Variables de entrada de todas las series en el origen t. Retorna X (series x 42) y la escala de cada serie."""
    reciente = Y[:, t - VENTANA + 1 : t + 1]
    escala = np.maximum(reciente.mean(axis=1, keepdims=True), 1)
    hace_un_anio = Y[:, t - ANIO + 1 : t - ANIO + 1 + H]
    base_anio = np.maximum(Y[:, t - ANIO - VENTANA + 1 : t - ANIO + 1].mean(axis=1, keepdims=True), 1)
    dia_semana = np.zeros((len(Y), 7), dtype="float32")
    dia_semana[:, (fechas[t].dayofweek + 1) % 7] = 1
    return np.hstack([reciente / escala, hace_un_anio / base_anio, dia_semana]), escala


def armar_tabla(origenes):
    """Apila las variables de varios orígenes. Retorna X y la escala de cada fila."""
    Xs, escalas = [], []
    for t in origenes:
        X, escala = armar_pistas(Y, t)
        Xs.append(X)
        escalas.append(escala)
    return np.vstack(Xs), np.vstack(escalas)


X_entrenamiento, _ = armar_tabla(np.concatenate([orig_train, orig_val]))
X_prueba, escala_prueba = armar_tabla(orig_test)
real = np.stack([Y[:, t + 1 : t + 1 + H] for t in orig_test])   # (semanas, series, días)
n_sem, n_prod = len(orig_test), len(Y)
print("entrenamiento:", X_entrenamiento.shape, "| prueba:", X_prueba.shape)

entrenamiento: (528500, 42) | prueba: (26000, 42)


In [4]:
def wape_semanal(pred):
    """Error absoluto del total semanal dividido por las ventas totales (métrica principal del notebook 02)."""
    return np.abs((pred - real).sum(axis=2)).sum() / real.sum()

## 2. Red float32 de referencia

Se carga la red de la versión v0.1 y se verifica que reproduce el error del notebook 02 antes de cuantizarla.

In [5]:
red = keras.models.load_model(f"../models/{VERSION_ORIGEN}/red.keras", compile=False)
red.summary()

pred_float32 = (red.predict(X_prueba, verbose=0) * escala_prueba).reshape(n_sem, n_prod, H)
wape_float32 = wape_semanal(pred_float32)
print(f"WAPE semanal float32: {wape_float32:.2%}")
assert abs(wape_float32 - 0.0528) < 0.002, "la red no reproduce el resultado del notebook 02"

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,063 (19.78 KB)

 Trainable params: 5,063 (19.78 KB)

 Non-trainable params: 0 (0.00 B)

WAPE semanal float32: 5.28%


## 3. Cuantización int8

La conversión usa un conjunto representativo de 1000 filas del período de entrenamiento para calibrar la escala y el punto cero de cada tensor. Se cuantizan pesos, activaciones, entrada y salida (cuantización entera completa), requisito de TensorFlow Lite Micro en microcontroladores.

In [6]:
carpeta_tmp = tempfile.mkdtemp()
red.export(carpeta_tmp, format="tf_saved_model", verbose=False)

rng = np.random.default_rng(42)
muestra = X_entrenamiento[rng.choice(len(X_entrenamiento), 1000, replace=False)]

def datos_representativos():
    for fila in muestra:
        yield [fila.reshape(1, -1)]

conversor = tf.lite.TFLiteConverter.from_saved_model(carpeta_tmp)
conversor.optimizations = [tf.lite.Optimize.DEFAULT]
conversor.representative_dataset = datos_representativos
conversor.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
conversor.inference_input_type = tf.int8
conversor.inference_output_type = tf.int8

modelo_int8 = conversor.convert()
print(f"Modelo int8: {len(modelo_int8):,} bytes")

INFO:tensorflow:Assets written to: C:\Users\davfy\AppData\Local\Temp\tmp7fsad3tt\assets


INFO:tensorflow:Assets written to: C:\Users\davfy\AppData\Local\Temp\tmp7fsad3tt\assets


Modelo int8: 10,144 bytes


## 4. Parámetros de cuantización

La entrada y la salida del modelo son enteros de 8 bits. El firmware debe aplicar la escala y el punto cero de la entrada antes de ejecutar el modelo, y los de la salida después.

In [7]:
interprete = tf.lite.Interpreter(model_content=modelo_int8)
interprete.allocate_tensors()
entrada = interprete.get_input_details()[0]
salida = interprete.get_output_details()[0]

esc_ent, cero_ent = entrada["quantization"]
esc_sal, cero_sal = salida["quantization"]
print(f"Entrada: forma {entrada['shape']}, escala {esc_ent:.6f}, punto cero {cero_ent}")
print(f"Salida:  forma {salida['shape']}, escala {esc_sal:.6f}, punto cero {cero_sal}")
print(f"Rango representable de entrada: {(-128 - cero_ent) * esc_ent:.2f} a {(127 - cero_ent) * esc_ent:.2f}")
print(f"Rango representable de salida:  {(-128 - cero_sal) * esc_sal:.2f} a {(127 - cero_sal) * esc_sal:.2f}")

Entrada: forma [ 1 42], escala 0.008903, punto cero -128
Salida:  forma [1 7], escala 0.005387, punto cero -128
Rango representable de entrada: 0.00 a 2.27
Rango representable de salida:  0.00 a 1.37


c:\Users\davfy\Documents\proeycts\tinyml-inventory-forecast\.venv\Lib\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [8]:
def predecir_int8(x):
    """Pronóstico normalizado de una fila (42 valores) con el modelo int8, igual que en el ESP32."""
    x_q = np.clip(np.round(x / esc_ent + cero_ent), -128, 127).astype(np.int8)
    interprete.set_tensor(entrada["index"], x_q.reshape(1, -1))
    interprete.invoke()
    y_q = interprete.get_tensor(salida["index"])[0]
    return (y_q.astype(np.float32) - cero_sal) * esc_sal


pred_int8 = np.vstack([predecir_int8(x) for x in X_prueba])
pred_int8 = (pred_int8 * escala_prueba).reshape(n_sem, n_prod, H)
wape_int8 = wape_semanal(pred_int8)

pd.DataFrame({
    "WAPE semanal": [f"{wape_float32:.2%}", f"{wape_int8:.2%}"],
    "Tamaño": [f"{red.count_params() * 4:,} bytes (pesos)", f"{len(modelo_int8):,} bytes (archivo)"],
}, index=["Red float32 (v0.1)", "Red int8 (v0.2)"])

,WAPE semanal,Tamaño
Red float32 (v0.1),5.28%,"20,252 bytes (pesos)"
Red int8 (v0.2),5.27%,"10,144 bytes (archivo)"


## 5. Exportación

Se guarda el modelo en formato TensorFlow Lite y como arreglo de bytes en `firmware/include/model_data.h`. La alineación de 16 bytes es un requisito de TensorFlow Lite Micro.

In [9]:
with open(f"{MODELOS_DIR}/red_int8.tflite", "wb") as f:
    f.write(modelo_int8)

os.makedirs(FIRMWARE_DIR, exist_ok=True)
with open(f"{FIRMWARE_DIR}/model_data.h", "w") as f:
    f.write("// Generado por ml/notebooks/03_cuantizacion_int8.ipynb. No editar a mano.\n")
    f.write("#pragma once\n\n")
    f.write("alignas(16) const unsigned char model_data[] = {\n")
    for i in range(0, len(modelo_int8), 12):
        bloque = modelo_int8[i : i + 12]
        f.write("  " + ", ".join(f"0x{b:02x}" for b in bloque) + ",\n")
    f.write("};\n")
    f.write(f"const unsigned int model_data_len = {len(modelo_int8)};\n")

print("Guardados:", f"{MODELOS_DIR}/red_int8.tflite", "y", f"{FIRMWARE_DIR}/model_data.h")

Guardados: ../models/v0.2/red_int8.tflite y ../../firmware/include/model_data.h


## 6. Conclusiones

- **Precisión.** La cuantización entera de 8 bits no degrada el modelo: el WAPE semanal sobre las 52 semanas de 2017 es 5.27 % en int8 frente a 5.28 % en float32. La diferencia se atribuye al redondeo.
- **Tamaño.** El archivo TensorFlow Lite ocupa 10,144 bytes. Los pesos pasan de 20,252 bytes en float32 a aproximadamente 5 KB en int8; el resto del archivo corresponde a la descripción de las capas y a los parámetros de cuantización.
- **Limitación.** El rango representable se fija con el conjunto representativo. La entrada admite valores entre 0 y 2.27 veces el promedio de los últimos 28 días, y la salida entre 0 y 1.37. Los valores superiores se saturan. Con el conjunto de Kaggle no se observa efecto sobre el error, pero con datos reales con picos de demanda pronunciados el pronóstico podría subestimarse; en ese caso se recalibraría la cuantización con datos propios.
- **Integración.** El modelo se exporta a `firmware/include/model_data.h` para su ejecución con TensorFlow Lite Micro. El firmware debe reproducir la construcción de las 42 variables de entrada (función `armar_pistas`) y aplicar la escala y el punto cero de la sección 4.

**Siguiente paso:** generar vectores de prueba (entradas y salidas esperadas) para verificar que el firmware obtiene los mismos resultados que el intérprete de Python.